# 06 - Key insights

**Audience:** a reader without a background in air quality or statistics.

**Question of the project:** is Seoul's air getting better, when and where is it most harmful, and can tomorrow's PM2.5 be predicted well enough to warn residents?

PM2.5 is fine dust: particles small enough to reach deep into the lungs. Ozone is a gas that forms in sunlight and irritates the airways. The data covers the 25 urban monitoring stations of Seoul, one per district, hour by hour from 2015 to 2025, from the official Seoul files.

Every number below is loaded from the files the earlier notebooks wrote to `reports/metrics/` and printed in the next cell.

## 1. Imports

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from seoul_air import config
from seoul_air.report import load_key_figures
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)
config.FIGURES.mkdir(parents=True, exist_ok=True)
config.METRICS.mkdir(parents=True, exist_ok=True)

## 2. The figures behind the findings

In [2]:
k = {nb: load_key_figures(nb) for nb in ["02", "03", "04", "05", "07", "08"]}
year, month, bad = k["02"]["pm25_mean_by_year"], k["02"]["pm25_median_by_month"], k["02"]["bad_or_worse_pct_by_month"]
pm25_trend, o3_trend = k["02"]["trend"]["pm25"], k["02"]["trend"]["o3"]
o3_above, cai_main = k["02"]["o3_above_8h_pct_by_year"], k["04"]["cai_main_pollutant_pct_by_year"]
share, groups, forecast = k["03"]["variance_share_pct"], k["03"]["categories"], k["05"]
episodes = k["04"]["episodes_by_year"]
consistent = groups.get("Above the rest of the city", 0) + groups.get("Below the rest of the city", 0)

facts = [
    ("Data", "Official 2024 city PM2.5: project vs Seoul figure (ug/m3)",
     f"{k['07']['city_pm25_by_year']['2024']:.2f} vs {k['07']['city_official']['2024_pm25']:g}"),
    ("Level", "Mean daily PM2.5 in 2016 / 2024 / 2025 (ug/m3)", f"{year['2016']:.1f} / {year['2024']:.1f} / {year['2025']:.1f}"),
    ("Trend", "PM2.5 change per year, with its 95% interval",
     f"{pm25_trend['pct_per_year']:.1f}% ({pm25_trend['pct_low_95']:.1f}% to {pm25_trend['pct_high_95']:.1f}%)"),
    ("Trend", "Stations whose PM2.5 declines clearly, of 25", f"{k['02']['stations_with_significant_pm25_decline']}"),
    ("Ozone", "Ozone change per year, with its 95% interval",
     f"{o3_trend['pct_per_year']:+.1f}% ({o3_trend['pct_low_95']:+.1f}% to {o3_trend['pct_high_95']:+.1f}%)"),
    ("Ozone", "Station-days above the 8-hour ozone standard, 2015 / 2024", f"{o3_above['2015']:.1f}% / {o3_above['2024']:.1f}%"),
    ("Ozone", "Hours in which ozone sets the air quality index (CAI), 2015 / 2025",
     f"{cai_main['2015']['o3']:.1f}% / {cai_main['2025']['o3']:.1f}%"),
    ("Season", "Median daily PM2.5, March / September (ug/m3)", f"{month['3']:.1f} / {month['9']:.1f}"),
    ("Season", "Station-days graded Bad or worse, March / August", f"{bad['3']:.1f}% / {bad['8']:.1f}%"),
    ("Episodes", "Episodes of 2 or more Bad days, 2016 / 2024", f"{episodes['2016']} / {episodes['2024']}"),
    ("Variation", "Share of the variation explained by the day / the district",
     f"{share['Date (each single day)']:.1f}% / {share['District']:.1f}%"),
    ("Districts", "Stations with a consistent difference / not comparable, of 25", f"{consistent} / {groups['Not comparable']}"),
    ("Exposure", "Largest change of the city PM2.5 when weighting by population (ug/m3)",
     f"{k['03']['exposure_pm25_max_abs_difference']:.2f}"),
    ("Roads", "NO2 at roadside stations compared with urban stations, 2024", f"{k['03']['roadside_vs_urban_pct_2024']['no2']:+.1f}%"),
    ("Weather", "Same-day association of PM2.5 with the mixing layer height (Spearman)",
     f"{k['08']['spearman_all_days']['boundary_layer_height_mean_m']:.2f}"),
    ("Forecast", "Next-day error of the model vs 'tomorrow = today' (2019, ug/m3)",
     f"{forecast['mae']['ridge']:.2f} vs {forecast['mae']['persistence']:.2f}"),
]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(facts, columns=["topic", "figure", "value"]).set_index(["topic", "figure"])

value
topic     figure                                                                                       
Data      Official 2024 city PM2.5: project vs Seoul figure (ug/m3)                       17.55 vs 17.6
Level     Mean daily PM2.5 in 2016 / 2024 / 2025 (ug/m3)                             26.2 / 17.5 / 18.5
Trend     PM2.5 change per year, with its 95% interval                           -3.5% (-4.5% to -2.5%)
          Stations whose PM2.5 declines clearly, of 25                                               25
Ozone     Ozone change per year, with its 95% interval                           +3.8% (+3.1% to +4.6%)
          Station-days above the 8-hour ozone standard, 2015 / 2024                       10.6% / 27.2%
          Hours in which ozone sets the air quality index (CAI), 2015 / 2025              17.2% / 46.5%
Season    Median daily PM2.5, March / September (ug/m3)                                     27.0 / 10.2
          Station-days graded Bad or worse, March / August                                 31.9% / 1.9%
Episodes  Episodes of 2 or more Bad days, 2016 / 2024                                            18 / 6
Variation Share of the variation explained by the day / the district                       94.4% / 0.3%
Districts Stations with a consistent difference / not comparable, of 25                          4 / 11
Exposure  Largest change of the city PM2.5 when weighting by population (ug/m3)                    0.17
Roads     NO2 at roadside stations compared with urban stations, 2024                            +41.2%
Weather   Same-day association of PM2.5 with the mixing layer height (Spearman)                   -0.39
Forecast  Next-day error of the model vs 'tomorrow = today' (2019, ug/m3)                  7.97 vs 8.86

![Monthly anomalies and long-term trends](../reports/figures/02_long_term_trend.png)

## 3. Key findings

1. **Fine dust has fallen by about a third, everywhere in the city.** The average day had 26.2 ug/m3 of PM2.5 in 2016 and 17.5 in 2024 (18.5 in 2025). Once the seasons are removed, the decline is 3.5% a year, and the uncertainty range (2.5% to 4.5%) is far from zero; all 25 stations decline. The air is still above the Korean annual standard of 15 ug/m3. The project's figures reproduce the official ones (17.55 against 17.6 for 2024).
2. **Ozone is moving the other way.** It rises by 3.8% a year. The share of station-days above the 8-hour ozone standard went from 10.6% in 2015 to 27.2% in 2024, and ozone now sets the official air quality index (CAI) in 46.5% of the hours, against 17.2% in 2015. In the warm months, ozone, not dust, is now the main problem.
3. **When matters far more than where.** The day explains 94.4% of the variation of PM2.5 and the district only 0.3%: bad air arrives across the whole city at once. March is the worst month (median 27.0 ug/m3, 31.9% of station-days Bad or worse) and September the cleanest (10.2). Multi-day episodes of bad air fell from 18 in 2016 to 6 in 2024. Days with a shallow mixing layer and little wind tend to have more PM2.5.
4. **Districts cannot be ranked, and population changes little.** Only 4 of the 25 stations keep a consistent difference from the rest of the city, and 11 were relocated or changed level during the period. Weighting each district by its population moves the city's PM2.5 by at most 0.17 ug/m3. Next to major roads the air is different: roadside stations record 41.2% more NO2 than the urban stations.
5. **Tomorrow's level can be predicted a little better than by assuming it equals today.** On the 2017-2019 data, the model's error is 7.97 ug/m3 against 8.86 for "tomorrow = today"; as an alert, it needs a lower trigger to catch as many Bad days.

## 4. Recommendations

These follow from the patterns in the data; they are not tested interventions.

- **Add ozone to public warnings.** Ozone exceedances concentrate in May to July and have more than doubled since 2015; warnings and advice for outdoor activity in the warm season should treat ozone as seriously as dust.
- **Keep the dust measures focused on December to March,** when Bad days and multi-day episodes concentrate, and communicate at city level, day by day: the day matters far more than the district.
- **Mind the roads.** People who spend hours next to major roads breathe more NO2 than the urban network shows; roadside stations should be part of exposure advice.
- **Do not compare districts without the station history.** Relocations and level shifts can move a station's level; month-level maintenance records from the operator would be needed to separate them from real differences.

## 5. Limitations and next steps

- **Trends are not adjusted for weather** and describe 2015-2025 rather than explain it; the COVID-19 period and the PM seasonal management system (from December 2019) are not separated yet.
- **One urban station per district**, and the official station history used here stops in 2019.
- **The forecast still uses the Kaggle data of 2017-2019**, and 2019 had been inspected before its protocol was fixed.
- **Weather is a reanalysis at one grid point**, with the mixing layer height missing for the first half of 2024.

**Next steps** (see `docs/ANALYSIS_PLAN.md`): a weather-adjusted trend and an evaluation of the seasonal management system, a 1-3 day forecast on the 2015-2025 data with weather forecasts, and scenario forecasts for the coming 2-3 years.